In [2]:
import pandas as pd
from pathlib import Path

DF_PATH = Path("../data/processed/online_retail_cleaned.parquet")
df = pd.read_parquet(DF_PATH)

In [3]:
# look at the dataframe
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


In [4]:
# info
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 504731 entries, 0 to 504730
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      504731 non-null  string        
 1   StockCode    504731 non-null  string        
 2   Description  504731 non-null  string        
 3   Quantity     504731 non-null  int16         
 4   InvoiceDate  504731 non-null  datetime64[us]
 5   Price        504731 non-null  float32       
 6   Customer ID  400916 non-null  Int64         
 7   Country      504731 non-null  category      
dtypes: Int64(1), category(1), datetime64[us](1), float32(1), int16(1), string(3)
memory usage: 41.3 MB


In [5]:
# revenue feature
df["Revenue"] = (df.Price * df.Quantity).astype("float32")

In [6]:
# create time features
df["Year"] = df.InvoiceDate.dt.year
df["Month"] = df.InvoiceDate.dt.month
df["Day"] = df.InvoiceDate.dt.day

In [7]:
df = df.copy()
snapshot_date = df.InvoiceDate.max()

In [8]:
# create customer-level dataset
customer_df = df.groupby("Customer ID").agg(
    Frequency=("Invoice", "nunique"),
    Monetary=("Revenue", "sum"),
    AvgQuantityPerLine=("Quantity", "mean"),
    TotalItems=("Quantity", "sum"),
    UniqueProducts=("StockCode", "nunique"),
    LastPurchase=("InvoiceDate", "max")
)

In [9]:
# compute how many days since last purchase
customer_df["Recency"] = (snapshot_date - customer_df.LastPurchase).dt.days

In [10]:
# drop helper column
customer_df = customer_df.drop(columns="LastPurchase")

In [11]:
# add behavioural features
customer_df["AvgOrderValue"] = (customer_df.Monetary / customer_df.Frequency)
customer_df["ItemsPerOrder"] = (customer_df.TotalItems / customer_df.Frequency)

In [12]:
# add geography
country_mode = df.groupby("Customer ID").Country.agg(lambda x: x.mode()[0])
customer_df["Country"] = country_mode.astype("category")

In [13]:
# clean final df
customer_df = customer_df.reset_index()

In [14]:
# check customer df
customer_df.head()

,Customer ID,Frequency,Monetary,AvgQuantityPerLine,TotalItems,UniqueProducts,Recency,AvgOrderValue,ItemsPerOrder,Country
0,12346,11,372.859985,2.121212,70,26,164,33.896362,6.363636,United Kingdom
1,12347,2,1323.320068,11.661972,828,70,2,661.660034,414.000000,Iceland
2,12348,1,222.160019,18.650000,373,20,73,222.160019,373.000000,Finland
3,12349,3,2671.139893,9.735294,993,90,42,890.379964,331.000000,Italy
4,12351,1,300.929993,12.428571,261,21,10,300.929993,261.000000,Unspecified


In [15]:
# info
customer_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4312 entries, 0 to 4311
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   Customer ID         4312 non-null   Int64   
 1   Frequency           4312 non-null   int64   
 2   Monetary            4312 non-null   float32 
 3   AvgQuantityPerLine  4312 non-null   float64 
 4   TotalItems          4312 non-null   int64   
 5   UniqueProducts      4312 non-null   int64   
 6   Recency             4312 non-null   int64   
 7   AvgOrderValue       4312 non-null   float64 
 8   ItemsPerOrder       4312 non-null   float64 
 9   Country             4312 non-null   category
dtypes: Int64(1), category(1), float32(1), float64(3), int64(4)
memory usage: 296.6 KB


In [16]:
# drop the Date now
df = df.drop("InvoiceDate", axis=1)

In [17]:
# check master df
df.head()

,Invoice,StockCode,Description,Quantity,Price,Customer ID,Country,Revenue,Year,Month,Day
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,6.95,13085,United Kingdom,83.399994,2009,12,1
1,489434,79323P,PINK CHERRY LIGHTS,12,6.75,13085,United Kingdom,81.000000,2009,12,1
2,489434,79323W,WHITE CHERRY LIGHTS,12,6.75,13085,United Kingdom,81.000000,2009,12,1
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2.10,13085,United Kingdom,100.799995,2009,12,1
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,1.25,13085,United Kingdom,30.000000,2009,12,1


In [18]:
# save customer df it
customer_df.to_parquet("../data/processed/customer_segmentation.parquet", index=False)

In [19]:
# save master df
df.to_parquet("../data/processed/online_retail_feature_engineered.parquet", index=False)